In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

LEADERBOARD_PATH = FYP_ROOT / 'aacbr' / 'leaderboard_trained.json'
print(f'Leaderboard: {LEADERBOARD_PATH}')
print(f'Exists: {LEADERBOARD_PATH.exists()}')

from aacbr.results_logger import load_leaderboard

df = load_leaderboard(LEADERBOARD_PATH)
print(f'\n{len(df)} entries loaded.')
print(f'eval_script counts:\n{df["eval_script"].value_counts().to_string()}')

In [ ]:
# ── Unify the two metric schemas into a single set of columns ─────────────────
#
# Older entries:  metrics.accuracy / metrics.ordinal_mae  (single train/test split)
# Newer entries:  metrics.cv_mean_accuracy / metrics.cv_mean_mae  (5-fold CV)
#
# We produce:
#   best_acc  -- CV mean if present, else single-split accuracy
#   best_f1   -- CV mean F1 if present, else NaN
#   best_mae  -- CV mean if present, else single-split MAE
#   acc_std / f1_std -- CV std if present, else NaN
#   eval_type -- 'cv' or 'split'
#   degenerate -- True if any bin is empty (results are meaningless)

def _col(df, name, default=np.nan):
    return df[name] if name in df.columns else pd.Series(default, index=df.index)

cv_acc  = _col(df, 'metrics.cv_mean_accuracy')
cv_f1   = _col(df, 'metrics.cv_mean_f1')
cv_mae  = _col(df, 'metrics.cv_mean_mae')
cv_std  = _col(df, 'metrics.cv_std_accuracy')
cv_f1s  = _col(df, 'metrics.cv_std_f1')
sp_acc  = _col(df, 'metrics.accuracy')
sp_mae  = _col(df, 'metrics.ordinal_mae')

df['best_acc']  = cv_acc.combine_first(sp_acc)
df['best_f1']   = cv_f1
df['best_mae']  = cv_mae.combine_first(sp_mae)
df['acc_std']   = cv_std
df['f1_std']    = cv_f1s
df['eval_type'] = np.where(cv_acc.notna(), 'cv', 'split')

# Flag entries where any bin is empty (degenerate quantile binning).
def _is_degenerate(row):
    dist_col = 'data_stats.class_distribution'
    if dist_col not in row or not isinstance(row[dist_col], list):
        return False
    return any(c == 0 for c in row[dist_col])

df['degenerate'] = df.apply(_is_degenerate, axis=1)

print('Metric type breakdown:')
print(df['eval_type'].value_counts().to_string())
print(f'\nDegenerate entries (empty bins): {df["degenerate"].sum()} of {len(df)}')
print('(These have artificially inflated accuracy and should not be compared against valid results.)')


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
FILTER_SCRIPT    = 'eval_trained_brainwear_2d'
FILTER_EVAL_TYPE = None

SUMMARY_COLS = [
    'run_id',
    'timestamp', 'eval_script', 'eval_type',
    'config.checkpoint',
    'config.score_name',
    'config.char_model', 'config.n_bins', 'config.agg_mode',
    'config.strategy', 'config.strict',
    'config.seed',
    'best_f1', 'f1_std', 'best_acc', 'acc_std', 'best_mae',
    'notes',
]

view = df.copy()
if FILTER_EVAL_TYPE:
    view = view[view['eval_type'] == FILTER_EVAL_TYPE]
if FILTER_SCRIPT:
    view = view[view['eval_script'] == FILTER_SCRIPT]

present = [c for c in SUMMARY_COLS if c in view.columns]
summary = view[present].copy()

for col in ['best_f1', 'f1_std', 'best_acc', 'acc_std', 'best_mae']:
    if col in summary.columns:
        summary[col] = summary[col].round(4)

if 'config.checkpoint' in summary.columns:
    summary['config.checkpoint'] = summary['config.checkpoint'].apply(
        lambda p: Path(p).parent.name if pd.notna(p) else p
    )

DEDUP_COLS = [c for c in [
    'eval_script', 'config.checkpoint', 'config.score_name',
    'config.char_model', 'config.n_bins', 'config.agg_mode',
    'config.strategy', 'config.strict', 'config.seed',
] if c in summary.columns]

summary = (
    summary
    .sort_values('timestamp', ascending=False)
    .drop_duplicates(subset=DEDUP_COLS, keep='first')
    .sort_values('best_f1', ascending=False)
    .reset_index(drop=True)
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 30)

type_label = FILTER_EVAL_TYPE or 'all'
print(f'{len(summary)} entries shown (eval_type={type_label}, script={FILTER_SCRIPT or "all"}).')
display_cols = [c for c in summary.columns if c != 'run_id']
summary[display_cols]


In [ ]:
# ── Best result per (eval_script, checkpoint, strategy) ───────────────────────
group_cols = [c for c in ['eval_script', 'config.checkpoint', 'config.strategy'] if c in df.columns]

best_per_strategy = (
    df[~df['degenerate']]
    .sort_values('best_f1', ascending=False)
    .groupby(group_cols, dropna=False)
    .first()
    .reset_index()
)

show_cols = group_cols + [c for c in [
    'config.char_model', 'config.n_bins', 'config.agg_mode',
    'config.strict', 'eval_type', 'best_f1', 'f1_std', 'best_acc', 'acc_std', 'best_mae',
] if c in best_per_strategy.columns]

result = best_per_strategy[show_cols].copy()
for col in ['best_f1', 'f1_std', 'best_acc', 'acc_std', 'best_mae']:
    if col in result.columns:
        result[col] = result[col].round(4)

if 'config.checkpoint' in result.columns:
    result['config.checkpoint'] = result['config.checkpoint'].apply(
        lambda p: Path(p).parent.name if pd.notna(p) else p
    )

result = result.sort_values(['eval_script', 'config.checkpoint', 'best_f1'], ascending=[True, True, False]).reset_index(drop=True)

print('Best result per (eval_script, checkpoint, strategy) -- degenerate entries excluded:')
result

In [ ]:
# ── Best result per (checkpoint, score_name) -- brainwear runs only ───────────
score_df = df[df['config.score_name'].notna() & ~df['degenerate']].copy()

if score_df.empty:
    print('No per-score entries found.')
else:
    group_cols_score = [c for c in ['config.checkpoint', 'config.score_name'] if c in score_df.columns]

    best_per_score = (
        score_df
        .sort_values('best_f1', ascending=False)
        .groupby(group_cols_score, dropna=False)
        .first()
        .reset_index()
    )

    show_cols_score = [c for c in [
        'config.checkpoint', 'config.score_name', 'eval_script', 'config.strategy',
        'config.char_model', 'config.n_bins', 'config.agg_mode',
        'config.strict', 'eval_type', 'best_f1', 'f1_std', 'best_acc', 'acc_std', 'best_mae',
    ] if c in best_per_score.columns]

    result_score = best_per_score[show_cols_score].copy()
    for col in ['best_f1', 'f1_std', 'best_acc', 'acc_std', 'best_mae']:
        if col in result_score.columns:
            result_score[col] = result_score[col].round(4)

    if 'config.checkpoint' in result_score.columns:
        result_score['config.checkpoint'] = result_score['config.checkpoint'].apply(
            lambda p: Path(p).parent.name if pd.notna(p) else p
        )

    result_score = result_score.sort_values(
        ['config.checkpoint', 'best_f1'], ascending=[True, False]
    ).reset_index(drop=True)

    print(f'Best result per (checkpoint, score_name) -- {len(result_score)} rows:')
    display(result_score)

In [ ]:
# ── Detail view: full metrics for one run ─────────────────────────────────────
INSPECT_IDX    = 1
INSPECT_RUN_ID = summary.iloc[INSPECT_IDX]['run_id'] if 'run_id' in summary.columns else None

with open(LEADERBOARD_PATH) as f:
    records = json.load(f)

record = next((r for r in records if r['run_id'] == INSPECT_RUN_ID), None)
if record is None:
    print(f'run_id {INSPECT_RUN_ID!r} not found')
else:
    print(f"run_id    : {record['run_id']}")
    print(f"timestamp : {record['timestamp']}")
    print(f"script    : {record['eval_script']}")
    print(f"notes     : {record.get('notes', '')}")
    print()
    print('Config:')
    for k, v in record['config'].items():
        print(f'  {k:<20} {v}')
    print()
    print('Data stats:')
    for k, v in record['data_stats'].items():
        print(f'  {k:<20} {v}')
    print()

    m = record['metrics']
    if 'cv_mean_f1' in m:
        print(f"CV F1       : {m['cv_mean_f1']:.4f} +/- {m['cv_std_f1']:.4f}")
        print(f"CV Accuracy : {m['cv_mean_accuracy']:.4f} +/- {m['cv_std_accuracy']:.4f}")
        print(f"CV MAE      : {m['cv_mean_mae']:.4f} +/- {m['cv_std_mae']:.4f}")
        print(f"Fold F1s    : {[round(x, 4) for x in m['fold_f1s']]}")
        print(f"Fold accs   : {[round(x, 4) for x in m['fold_accs']]}")
        print(f"Fold MAEs   : {[round(x, 4) for x in m['fold_maes']]}")
    elif 'cv_mean_accuracy' in m:
        print(f"CV Accuracy : {m['cv_mean_accuracy']:.4f} +/- {m['cv_std_accuracy']:.4f}")
        print(f"CV MAE      : {m['cv_mean_mae']:.4f} +/- {m['cv_std_mae']:.4f}")
        print(f"Fold accs   : {[round(x, 4) for x in m['fold_accs']]}")
        print(f"Fold MAEs   : {[round(x, 4) for x in m['fold_maes']]}")
    else:
        print(f"Accuracy    : {m['accuracy']:.4f}")
        print(f"Ordinal MAE : {m['ordinal_mae']:.4f}")
        if 'confusion_matrix' in m:
            print()
            print('Confusion matrix (rows=true, cols=pred):')
            print(np.array(m['confusion_matrix']))
        if 'per_class' in m:
            print()
            print('Per-class metrics:')
            print(pd.DataFrame(m['per_class']).T.round(3))


In [ ]:
# ── Multi-bar chart: best F1 per (eval_script, strategy), colours = model ──
import matplotlib.pyplot as plt

if len(best_per_strategy) == 0:
    print('No runs to plot.')
else:
    eval_scripts = sorted(best_per_strategy['eval_script'].unique())
    strategies   = sorted(best_per_strategy['config.strategy'].unique())
    all_ckpts    = sorted(best_per_strategy['config.checkpoint'].dropna().unique())
    palette      = plt.cm.tab10.colors
    ckpt_color   = {c: palette[i % 10] for i, c in enumerate(all_ckpts)}
    ckpt_short   = {c: Path(c).parent.name if pd.notna(c) else str(c) for c in all_ckpts}

    n_scripts = len(eval_scripts)
    fig, axes = plt.subplots(n_scripts, 1,
                             figsize=(max(8, 5 * len(strategies)), 5 * n_scripts),
                             squeeze=False)

    for row, script in enumerate(eval_scripts):
        ax     = axes[row][0]
        sub    = best_per_strategy[best_per_strategy['eval_script'] == script]
        ckpts  = sorted(sub['config.checkpoint'].dropna().unique())
        n_c    = len(ckpts)
        width  = 0.8 / max(n_c, 1)
        x      = np.arange(len(strategies))

        for i, ckpt in enumerate(ckpts):
            ckpt_sub = (sub[sub['config.checkpoint'] == ckpt]
                        .set_index('config.strategy'))
            heights = [float(ckpt_sub.loc[s, 'best_f1'])
                       if s in ckpt_sub.index and pd.notna(ckpt_sub.loc[s, 'best_f1'])
                       else np.nan for s in strategies]
            errs = [float(ckpt_sub.loc[s, 'f1_std'])
                    if s in ckpt_sub.index and pd.notna(ckpt_sub.loc[s, 'f1_std'])
                    else 0.0 for s in strategies] if 'f1_std' in sub.columns else None
            offset = (i - (n_c - 1) / 2) * width
            ax.bar(x + offset, heights, width,
                   label=ckpt_short[ckpt], color=ckpt_color[ckpt],
                   yerr=errs, capsize=3, alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(strategies, rotation=30, ha='right', fontsize=9)
        ax.set_ylabel('Macro-F1')
        ax.set_ylim(0, 1)
        ax.set_title(script.replace('eval_trained_', ''))
        ax.legend(fontsize=8, framealpha=0.8)
        ax.grid(axis='y', ls=':', alpha=0.5)

    plt.suptitle('Best F1 per (eval_script, strategy) — colours = model checkpoint',
                 fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Multi-bar charts: best CV F1 per strategy, separate figures for brats / brainwear
import matplotlib.pyplot as plt
import numpy as np

n_bins_vals = sorted(df['config.n_bins'].dropna().unique())
strategies  = sorted(df['config.strategy'].dropna().unique())
palette     = plt.cm.tab10.colors

SCRIPTS = [
    ('eval_trained_2d',           'BraTS'),
    ('eval_trained_brainwear_2d', 'BrainWear'),
]

for script_name, script_label in SCRIPTS:
    script_df = df[(df['eval_script'] == script_name) & ~df['degenerate']]
    if script_df.empty:
        print(f'No entries for {script_name}')
        continue

    ckpts      = sorted(script_df['config.checkpoint'].dropna().unique())
    ckpt_color = {c: palette[i % 10] for i, c in enumerate(ckpts)}
    ckpt_short = {c: Path(c).parent.name for c in ckpts}
    n_c        = len(ckpts)
    width      = 0.8 / max(n_c, 1)

    fig, axes = plt.subplots(len(n_bins_vals), 1,
                             figsize=(max(6, 4 * len(strategies)), 5 * len(n_bins_vals)),
                             squeeze=False)

    for ax, n_bins in zip(axes[:, 0], n_bins_vals):
        sub = script_df[script_df['config.n_bins'] == n_bins]
        x   = np.arange(len(strategies))

        for i, ckpt in enumerate(ckpts):
            ckpt_sub = (
                sub[sub['config.checkpoint'] == ckpt]
                .sort_values('best_f1', ascending=False)
                .groupby('config.strategy')
                .first()
                .reindex(strategies)
            )
            heights = ckpt_sub['best_f1'].fillna(np.nan).values
            errs    = ckpt_sub['f1_std'].fillna(0.0).values if 'f1_std' in ckpt_sub.columns else np.zeros(len(strategies))
            offset  = (i - (n_c - 1) / 2) * width

            bars = ax.bar(x + offset, heights, width,
                          label=ckpt_short[ckpt], color=ckpt_color[ckpt],
                          yerr=errs, capsize=3, alpha=0.85)
            for bar, h in zip(bars, heights):
                if not np.isnan(h):
                    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                            f'{h:.3f}', ha='center', va='bottom', fontsize=7)

        ax.set_xticks(x)
        ax.set_xticklabels(strategies, rotation=30, ha='right', fontsize=9)
        ax.set_ylabel('Best CV F1')
        ax.set_ylim(0, 1.05)
        ax.set_title(f'n_bins = {int(n_bins)}')
        ax.axhline(1 / n_bins, color='grey', linestyle=':', linewidth=1.2,
                   label=f'chance (1/{int(n_bins)})')
        ax.legend(fontsize=8)
        ax.grid(axis='y', ls=':', alpha=0.5)

    plt.suptitle(f'{script_label} — Best CV F1 per strategy, grouped by n_bins — colours = model',
                 fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── BrainWear multi-bar chart filtered by outcome score ───────────────────────
# Set FILTER_SCORE_NAME to one of the score names (e.g. 'CF', 'BNFU', 'PF2',
# 'EF', 'QL2') or None to show all scores combined.
FILTER_SCORE_NAME = 'PF2'   # <── change this

import matplotlib.pyplot as plt
import numpy as np

script_name = 'eval_trained_brainwear_2d'
script_df   = df[(df['eval_script'] == script_name) & ~df['degenerate']].copy()

if FILTER_SCORE_NAME is not None:
    script_df = script_df[script_df['config.score_name'] == FILTER_SCORE_NAME]

if script_df.empty:
    available = df[df['eval_script'] == script_name]['config.score_name'].dropna().unique().tolist()
    print(f'No entries for score_name={FILTER_SCORE_NAME!r}.  Available: {available}')
else:
    score_label = FILTER_SCORE_NAME if FILTER_SCORE_NAME else 'all scores'
    n_bins_vals = sorted(script_df['config.n_bins'].dropna().unique())
    strategies  = sorted(script_df['config.strategy'].dropna().unique())
    palette     = plt.cm.tab10.colors

    ckpts      = sorted(script_df['config.checkpoint'].dropna().unique())
    ckpt_color = {c: palette[i % 10] for i, c in enumerate(ckpts)}
    ckpt_short = {c: Path(c).parent.name for c in ckpts}
    n_c        = len(ckpts)
    width      = 0.8 / max(n_c, 1)

    fig, axes = plt.subplots(len(n_bins_vals), 1,
                             figsize=(max(6, 4 * len(strategies)), 5 * len(n_bins_vals)),
                             squeeze=False)

    for ax, n_bins in zip(axes[:, 0], n_bins_vals):
        sub = script_df[script_df['config.n_bins'] == n_bins]
        x   = np.arange(len(strategies))

        for i, ckpt in enumerate(ckpts):
            ckpt_sub = (
                sub[sub['config.checkpoint'] == ckpt]
                .sort_values('best_f1', ascending=False)
                .groupby('config.strategy')
                .first()
                .reindex(strategies)
            )
            heights = ckpt_sub['best_f1'].fillna(np.nan).values
            errs    = ckpt_sub['f1_std'].fillna(0.0).values if 'f1_std' in ckpt_sub.columns else np.zeros(len(strategies))
            offset  = (i - (n_c - 1) / 2) * width

            bars = ax.bar(x + offset, heights, width,
                          label=ckpt_short[ckpt], color=ckpt_color[ckpt],
                          yerr=errs, capsize=3, alpha=0.85)
            for bar, h in zip(bars, heights):
                if not np.isnan(h):
                    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                            f'{h:.3f}', ha='center', va='bottom', fontsize=7)

        ax.set_xticks(x)
        ax.set_xticklabels(strategies, rotation=30, ha='right', fontsize=9)
        ax.set_ylabel('Best CV F1')
        ax.set_ylim(0, 1.05)
        ax.set_title(f'n_bins = {int(n_bins)}')
        ax.axhline(1 / n_bins, color='grey', linestyle=':', linewidth=1.2,
                   label=f'chance (1/{int(n_bins)})')
        ax.legend(fontsize=8)
        ax.grid(axis='y', ls=':', alpha=0.5)

    plt.suptitle(f'BrainWear [{score_label}] — Best CV F1 per strategy, colours = model checkpoint',
                 fontsize=13)
    plt.tight_layout()
    plt.show()
